# Results comparison: periods 1–4

Reads `results_period_1` through `results_period_4` beside this notebook. Run all cells from top to bottom. Set `RESULTS_ROOT` in the configuration cell if needed.

Emissions and transport use `time_weights.csv` to calculate annual totals. Captured CO₂ is excluded from emissions; negative emissions remain visible. Material capacity is converted from tonnes/hour to million tonnes/year using 8,760 hours; electricity and hydrogen capacities remain MW. Storage plots use discharge capacity, excluding charging and stored energy. Existing-capacity maps show the beginning of period 1.

Sectors are selected from commodity and resource-type metadata; unfamiliar technology names are retained. `capacity_asset_inventory` contains all exported capacity rows, and `other_capacity_summary` reports raw capacities by commodity and asset type (units vary; do not add across commodities). Biomass/BECCS capacity is exported as biomass input, so it is listed separately rather than added to electrical output MW. Transport summaries count each link, including multiple links used by the same material.

**CO₂ cap coverage:** these saved results were generated with ammonia and methanol emissions routed to regional nodes with caps disabled. The asset inputs have since been corrected to use the national sink; rerun the optimization to update the results. The emissions chart includes these emissions and therefore differs from the capped national-node total. Biomass/BECCS net emissions include both CO₂ uptake and `co2_emission_edge` releases.


In [ ]:
import pandas as pd
import numpy as np
import re
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from functools import reduce
from pathlib import Path
import warnings

In [ ]:
# Run from this notebook's folder, or set RESULTS_ROOT explicitly.
RESULTS_ROOT = Path.cwd()
if not (RESULTS_ROOT / "results_period_1").is_dir():
    matches = list(RESULTS_ROOT.glob("**/with_cap/results_period_1"))
    if len(matches) != 1:
        raise FileNotFoundError("Set RESULTS_ROOT to the with_cap folder containing results_period_1–4.")
    RESULTS_ROOT = matches[0].parent
cases = [f"period_{i}" for i in range(1, 5)]
scenario_map = {case: f"Period {i}" for i, case in enumerate(cases, 1)}
REGION_PATTERN = re.compile(r"R?Region\d+([A-Za-z]+)", re.I)

def technology_id(value):
    # Handles both Region1Beijing_asset and asset_Region1Beijing.
    return REGION_PATTERN.sub("", str(value)).strip("_")

all_flow_dfs, all_capacity_dfs, time_weights = {}, {}, {}
for case in cases:
    folder = RESULTS_ROOT / f"results_{case}"
    flows = pd.read_csv(folder / "flows.csv")
    weights = pd.read_csv(folder / "time_weights.csv")
    if flows["time"].duplicated().any() or weights["time"].duplicated().any():
        raise ValueError(f"Duplicate time values in {case}")
    aligned = flows["time"].map(weights.set_index("time")["weight"])
    if aligned.isna().any() or not np.isfinite(aligned).all() or (aligned <= 0).any():
        raise ValueError(f"Missing or invalid time weights in {case}")
    time_weights[case] = aligned
    # Store annual totals as one row. All downstream sums now use TDR weights.
    all_flow_dfs[case] = flows.drop(columns="time").mul(aligned, axis=0).sum().to_frame().T
    capacity = pd.read_csv(folder / "capacity.csv")
    capacity["region_name"] = capacity["resource_id"].str.extract(REGION_PATTERN, expand=False)
    capacity["technology"] = capacity["resource_id"].map(technology_id)
    all_capacity_dfs[case] = capacity
    print(f"{scenario_map[case]}: {len(capacity):,} capacity rows, {len(flows)} timesteps, {aligned.sum():g} weighted hours")

In [ ]:
def get_flow_sums_by_tech(flow_df, include_keywords=None, drop_techs=None):
    """Region × technology annual CO2 emissions; captured CO2 is excluded."""
    rows = []
    for col, value in flow_df.sum().items():
        if not re.search(r"_co2(?:_emissions?)?_edge$", col, re.I):
            continue
        match = REGION_PATTERN.search(col)
        tech = technology_id(col)
        if drop_techs and tech in drop_techs:
            continue
        rows.append({"Region": match.group(1) if match else "Unassigned",
                     "Technology": tech, "Value": value})
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).pivot_table(index="Region", columns="Technology",
                                         values="Value", aggfunc="sum", fill_value=0)

In [ ]:
tech_name_map = {
    "oxyfuelcement_retrofit_co2_emissions_edge": "Oxyfuel Cement Retrofit",
    "meacement_co2_emissions_edge": "MEA Cement Retrofit",
    "tradcement_co2_emissions_edge": "Traditional Cement",
    "bfbof_co2_edge": "BF-BOF",
    "bfbofccs_co2_edge": "BF-BOF-CCS",
    "bfbofccs_retrofit_co2_edge": "BF-BOF-CCS Retrofit",
    "scrapeaf56_co2_edge": "Scrap EAF 56%",
    "scrapeaf100_co2_edge": "Scrap EAF 100%",
    "scrapeaf100_retrofit_co2_edge": "Scrap EAF 100% Retrofit",
    "drieaf_co2_edge": "DRI-EAF",
    "drieaf_retrofit_co2_edge": "DRI-EAF Retrofit",
    "gasdri_co2_edge": "Gas DRI",
    "gasdriccs_co2_edge": "Gas DRI with CCS",
    "h2dri_co2_edge": "H2 DRI",
    "aluminum_smelting_co2_edge": "Aluminum Smelting",
    "coal_co2_edge": "Coal Power",
    "naturalgas_power_co2_edge": "Natural Gas Power",
}

custom_color_map = {
    "Oxyfuel Cement Retrofit": "#ff9d9a",
    "MEA Cement Retrofit": "#f28e9b",
    "Traditional Cement": "#e15759",
    "BF-BOF": "#1f77b4",
    "BF-BOF-CCS": "#0ee3ff",
    "BF-BOF-CCS Retrofit": "#aec7e8",
    "Scrap EAF 56%": "#38a938",
    "Scrap EAF 100%": "#6acaa2",
    "Scrap EAF 100% Retrofit": "#98df8a",
    "DRI-EAF": "#9467bd",
    "DRI-EAF Retrofit": "#c5b0d5",
    "Gas DRI": "#8c564b",
    "Gas DRI with CCS": "#e377c2",
    "H2 DRI": "#17becf",
    "Aluminum Smelting": "#bbbbb2",
    "Natural Gas Power": "#eb7a23",
    "Coal Power": "#4a5568",
    "Hydropower": "#3e6496",
    "Solar PV (Centralized)": "#fbb04e",
    "Solar PV (Distributed)": "#f1ce63",
    "Wind (Offshore)": "#638fbb",
    "Wind (Onshore)": "#90bfe2",
    "Battery Storage": "#9f6490",
    "Nuclear": "#3c8782",
    "Pumped Hydro Storage": "#75afa7",
}

In [ ]:
combined_co2_by_tech = pd.DataFrame({
    case: get_flow_sums_by_tech(all_flow_dfs[case]).sum() for case in cases
}).fillna(0)
combined_scenario_map = scenario_map
# Preserve every technology, including technologies first appearing in later periods.
for tech in combined_co2_by_tech.index:
    tech_name_map.setdefault(tech, re.sub(r"_co2(?:_emissions?)?_edge$", "", tech, flags=re.I).replace("_", " "))
for i, label in enumerate(sorted(set(tech_name_map.values()))):
    custom_color_map.setdefault(label, px.colors.qualitative.Alphabet[i % 26])
combined_co2_by_tech

In [ ]:
def plot_stacked_bar(
    df,
    scenario_map=None,
    name_map=None,
    color_map=None,
    title_name="none",
    xaxis_name="none",
    yaxis_name="none",
    width=1200,
    height=600,
    ylim=None,
):
    if scenario_map:
        df = df.rename(index=scenario_map)

    df = df.T

    if not df.index.name:
        df = df.copy()
        df.index.name = "Category"

    row_label = df.index.name

    df_long = df.reset_index().melt(
        id_vars=row_label,
        var_name="Scenario",
        value_name="Value"
    )

    display_col = row_label + "_Display"

    if name_map:
        df_long[display_col] = df_long[row_label].map(lambda x: name_map.get(x, x))
    else:
        df_long[display_col] = df_long[row_label]

    if color_map and name_map:
        color_map = {name_map.get(k, k): v for k, v in color_map.items()}

    fig = px.bar(
        df_long,
        x="Scenario",
        y="Value",
        color=display_col,
        title=None if title_name == "none" else title_name,
        labels={
            "Scenario": xaxis_name if xaxis_name != "none" else "Scenario",
            "Value": yaxis_name if yaxis_name != "none" else "Value",
            display_col: row_label
        },
        color_discrete_map=color_map,
    )

    fig.update_layout(
        barmode="relative",
        template="plotly_white",
        width=width,
        height=height,
        margin=dict(l=60, r=60, t=80, b=80),
        legend=dict(traceorder="reversed")
    )

    fig.update_xaxes(tickangle=-45)

    if ylim is not None:
        fig.update_yaxes(range=ylim)

    return fig

In [ ]:
def plot_scenario_subplots(
    co2_dict,
    scenario_map,
    name_map=None,
    color_map=None,
    ymax=30e6,
    ncols=None,
    title_prefix="",
    xaxis_name="none",
    yaxis_name="none",
    width=2500
):
    keys = list(co2_dict.keys())
    n = len(keys)

    if ncols is None:
        ncols = n

    fig = make_subplots(
        rows=1,
        cols=ncols,
        subplot_titles=[f"{title_prefix}{scenario_map[k]}" for k in keys],
        horizontal_spacing=0.05,
    )

    for col_idx, key in enumerate(keys, start=1):
        df = co2_dict[key]

        subfig = plot_stacked_bar(
            df,
            scenario_map=scenario_map,
            name_map=name_map,
            color_map=color_map,
            title_name="none",
            xaxis_name="none",
            yaxis_name="none"
        )

        for trace in subfig.data:
            trace.showlegend = (col_idx == 1)
            fig.add_trace(trace, row=1, col=col_idx)

        fig.update_yaxes(range=[0, ymax], row=1, col=col_idx)

    if yaxis_name != "none":
        fig.update_yaxes(title_text=yaxis_name, row=1, col=1)

    if xaxis_name != "none":
        for col_idx in range(1, ncols + 1):
            fig.update_xaxes(title_text=xaxis_name, row=1, col=col_idx)

    fig.update_layout(
        template="plotly_white",
        barmode="relative",
        height=650,
        width=width,
        legend_title="Category",
        margin=dict(l=40, r=40, t=80, b=80),
        legend=dict(traceorder="reversed")
    )

    fig.update_xaxes(tickangle=-45)

    return fig

# CO2 Emissions

In [ ]:
plot_stacked_bar(
    combined_co2_by_tech.T, scenario_map=scenario_map,
    name_map=tech_name_map, color_map=custom_color_map,
    title_name="National CO2 Emissions by Period", xaxis_name="Period",
    yaxis_name="Annual CO2 Emissions (Tonnes)", width=1000,
).show()

# Capacity Analysis

In [ ]:
# Commodity/type metadata selects sectors independently of resource naming.
SECTOR_COMMODITIES = {
    "electricity": "Electricity", "cement": "Cement", "steel": "CrudeSteel",
    "aluminum": "Aluminum", "dri": "DRI", "alumina": "Alumina",
    "ammonia": "Ammonia", "methanol": "Methanol", "hydrogen": "Hydrogen",
}
CAPACITY_LABELS = {
    "coal": "Coal Power", "naturalgaspower": "Natural Gas Power",
    "naturalgas_power": "Natural Gas Power", "coalpowerccs": "Coal Power CCS",
    "gaspowerccs": "Natural Gas Power CCS", "hydropower": "Hydropower",
    "utility_scale_solar_pv": "Solar PV (Centralized)", "pv_central": "Solar PV (Centralized)",
    "distributed_solar_pv": "Solar PV (Distributed)", "pv_distribution": "Solar PV (Distributed)",
    "offshore_wind": "Wind (Offshore)", "wind_offshore": "Wind (Offshore)",
    "onshore_wind": "Wind (Onshore)", "wind_onshore": "Wind (Onshore)",
    "battery": "Battery Storage", "nuclear": "Nuclear", "pumpedhydro": "Pumped Hydro Storage",
}
for key, label in tech_name_map.items():
    CAPACITY_LABELS.setdefault(re.sub(r"_co2(?:_emissions?)?_edge$", "", key, flags=re.I).lower(), label)
CAPACITY_LABELS.update({"meacement": "MEA Cement", "meacement_retrofit": "MEA Cement Retrofit",
                        "oxyfuelcement": "Oxyfuel Cement", "aluminum_refining": "Aluminum Refining"})

def capacity_rows(df, commodity):
    # Discharge power only: exclude charging, reservoir energy and transmission.
    ids = df.component_id.str.lower()
    return df.loc[
        df.commodity.eq(commodity)
        & ~df.resource_type.str.contains("Transmission|GasStorage", case=False, na=False)
        & ~ids.str.contains(r"(?:_charge_edge|_inflow_edge|_storage)$", regex=True)
    ].copy()

def extract_capacity(df, commodity, column="capacity"):
    selected = capacity_rows(df, commodity)
    selected["tech"] = selected.technology.map(lambda t: CAPACITY_LABELS.get(t.lower(), t.replace("_", " ")))
    if selected.region_name.isna().any():
        warnings.warn(f"Unassigned regions in {commodity} capacity; see capacity_asset_inventory")
    selected["region_name"] = selected.region_name.fillna("Unassigned")
    return selected.pivot_table(index="region_name", columns="tech", values=column,
                                aggfunc="sum", fill_value=0)

def consolidate_capacity(sector_dict):
    return pd.DataFrame({case: df.sum() for case, df in sector_dict.items()}).fillna(0)

def build_combined_capacity(sector_dict):
    return consolidate_capacity(sector_dict), scenario_map

In [ ]:
sector_capacity, sector_existing = {}, {}
for sector, commodity in SECTOR_COMMODITIES.items():
    # Electricity and hydrogen are energy rates (MW); industrial materials are t/h.
    scale = 1 if sector in ("electricity", "hydrogen") else 8760 / 1e6
    sector_capacity[sector] = {c: extract_capacity(all_capacity_dfs[c], commodity) * scale for c in cases}
    sector_existing[sector] = {c: extract_capacity(all_capacity_dfs[c], commodity, "existing_capacity") * scale for c in cases}
    technologies = sorted(set().union(*(df.columns for df in sector_capacity[sector].values())))
    for c in cases:
        for collection in (sector_capacity, sector_existing):
            collection[sector][c] = collection[sector][c].reindex(columns=technologies, fill_value=0)
    for label in technologies:
        custom_color_map.setdefault(label, px.colors.qualitative.Alphabet[len(custom_color_map) % 26])
all_elec_dfs, all_cement_dfs, all_steel_dfs, all_aluminum_dfs, all_dri_dfs = (
    sector_capacity[s] for s in ("electricity", "cement", "steel", "aluminum", "dri"))
all_elec_ex_dfs, all_cement_ex_dfs, all_steel_ex_dfs, all_aluminum_ex_dfs, all_dri_ex_dfs = (
    sector_existing[s] for s in ("electricity", "cement", "steel", "aluminum", "dri"))
capacity_asset_inventory = pd.concat([
    df.assign(period=scenario_map[c]) for c, df in all_capacity_dfs.items()
], ignore_index=True)
# Biomass power is fuel-input-rated in capacity.csv, so keep it separate from MW electricity output.
other_capacity_summary = capacity_asset_inventory.groupby(
    ["period", "commodity", "resource_type"], dropna=False)["capacity"].sum().unstack("period")
print("Capacity extracted for:", ", ".join(sector_capacity))
other_capacity_summary

In [ ]:
# Electricity capacity — four-period comparison
combined_elec, combined_elec_smap = build_combined_capacity(all_elec_dfs)

plot_stacked_bar(
    combined_elec.T,
    scenario_map=combined_elec_smap,
    name_map=tech_name_map,
    color_map=custom_color_map,
    title_name="Electricity Capacity — Periods 1–4",
    xaxis_name="Period",
    yaxis_name="National Electricity Capacity (MW)",
    width=1200
)

In [ ]:
# Steel capacity — four-period comparison
combined_steel, combined_steel_smap = build_combined_capacity(all_steel_dfs)

plot_stacked_bar(
    combined_steel.T,
    scenario_map=combined_steel_smap,
    name_map=tech_name_map,
    color_map=custom_color_map,
    title_name="Steel Capacity — Periods 1–4",
    xaxis_name="Period",
    yaxis_name="National Steel Production Capacity (MT / year)",
    width=1200
)

In [ ]:
# Cement capacity — four-period comparison
combined_cement, combined_cement_smap = build_combined_capacity(all_cement_dfs)

plot_stacked_bar(
    combined_cement.T,
    scenario_map=combined_cement_smap,
    name_map=tech_name_map,
    color_map=custom_color_map,
    title_name="Cement Capacity — Periods 1–4",
    xaxis_name="Period",
    yaxis_name="National Cement Production Capacity (MT / year)",
    width=1200
)

In [ ]:
# Aluminum capacity — four-period comparison
combined_aluminum, combined_aluminum_smap = build_combined_capacity(all_aluminum_dfs)

plot_stacked_bar(
    combined_aluminum.T,
    scenario_map=combined_aluminum_smap,
    name_map=tech_name_map,
    color_map=custom_color_map,
    title_name="Aluminum Capacity — Periods 1–4",
    xaxis_name="Period",
    yaxis_name="National Aluminum Production Capacity (MT / year)",
    width=1200
)

In [ ]:
# DRI capacity — four-period comparison
combined_dri, combined_dri_smap = build_combined_capacity(all_dri_dfs)

plot_stacked_bar(
    combined_dri.T,
    scenario_map=combined_dri_smap,
    name_map=tech_name_map,
    color_map=custom_color_map,
    title_name="DRI-Making Capacity — Periods 1–4",
    xaxis_name="Period",
    yaxis_name="National DRI Capacity (MT / year)",
    width=1200
)

# Pie Charts

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Wedge, Circle

def plot_province_pies(
    gdf,
    data_df,
    province_col="NAME_1",
    categories=None,
    colors=None,
    pie_scale=1.0,
    figsize=(14, 10),
    dpi=150,
    title="",
    legend_sizes=None,
    capacity_unit="MW",
    max_total_override=None,
    ax=None,
):
    if categories is None:
        categories = list(data_df.columns)

    if colors is None:
        cmap = plt.cm.tab20
        colors = {cat: cmap(i) for i, cat in enumerate(categories)}

    gdf_proj = gdf.to_crs(epsg=3857)
    gdf_proj = gdf_proj.merge(data_df, left_on=province_col, right_index=True, how="left")
    gdf_proj[categories] = gdf_proj[categories].fillna(0)

    gdf_proj["centroid"] = gdf_proj.geometry.centroid
    gdf_proj["cx"] = gdf_proj.centroid.x
    gdf_proj["cy"] = gdf_proj.centroid.y

    gdf_proj["total_value"] = gdf_proj[categories].sum(axis=1)
    max_total = max_total_override if max_total_override is not None else (gdf_proj["total_value"].max() or 1)
    gdf_proj["radius"] = (gdf_proj["total_value"] / max_total) * pie_scale * 80_000

    _own_fig = ax is None
    if _own_fig:
        fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    ax.set_axis_off()

    for _, row in gdf_proj.iterrows():
        cx, cy, r = row["cx"], row["cy"], row["radius"]
        vals = row[categories].values.astype(float)
        total = vals.sum()
        if total == 0:
            continue
        angles = np.cumsum(vals) / total * 360
        prev_angle = 0
        for cat, angle in zip(categories, angles):
            wedge = Wedge(
                center=(cx, cy), r=r,
                theta1=prev_angle, theta2=angle,
                facecolor=colors[cat], edgecolor="none",
                alpha=1.0, zorder=1000
            )
            ax.add_patch(wedge)
            prev_angle = angle

    gdf_proj.plot(ax=ax, color="#F4F4F4", edgecolor="none", zorder=0)
    gdf_proj.boundary.plot(ax=ax, linewidth=0.3, color="black", zorder=1)

    if legend_sizes is not None and len(legend_sizes) > 0:
        legend_radii = [(s / max_total) * pie_scale * 80_000 for s in legend_sizes]
        sizes_and_radii = sorted(zip(legend_sizes, legend_radii))
        max_r = max(legend_radii)

        xlim = ax.get_xlim()
        ylim = ax.get_ylim()
        plot_width = xlim[1] - xlim[0]
        pad_x = plot_width * 0.02
        gap = max_r * 0.5

        # Expand ylim downward to place legend below the map, never overlapping it
        strip_height = gap + sum(r * 2 + gap for _, r in sizes_and_radii)
        ax.set_ylim(ylim[0] - strip_height, ylim[1])
        ylim = ax.get_ylim()

        legend_cx = xlim[0] + pad_x + max_r
        y = ylim[0] + gap
        for size, radius in sizes_and_radii:
            cy_legend = y + radius
            circle = Circle(
                (legend_cx, cy_legend), radius,
                facecolor="white", edgecolor="#555555",
                linewidth=1.2, zorder=2000
            )
            ax.add_patch(circle)
            ax.text(
                legend_cx + max_r + pad_x * 0.6, cy_legend,
                f"{size:,.0f} {capacity_unit}",
                va="center", ha="left", fontsize=9, zorder=2001
            )
            y += radius * 2 + gap

    if _own_fig:
        plt.title(title, fontsize=18)
        plt.tight_layout()
        plt.show()
        plt.close()

    return ax

In [ ]:
GEODATA_PATH = RESULTS_ROOT.parent / "plotting_software" / "data" / "gadm36_CHN_1.json"
gdf = gpd.read_file(GEODATA_PATH)
rename_map = {"Nei Mongol": "Innermongolia", "Xizang": "Tibet", "Ningxia Hui": "Ningxia", "Xinjiang Uygur": "Xinjiang"}
gdf["NAME_1"] = gdf["NAME_1"].replace(rename_map)
missing_regions = set(capacity_asset_inventory.region_name.dropna()) - set(gdf.NAME_1)
if missing_regions:
    warnings.warn(f"Capacity regions missing from map: {sorted(missing_regions)}")

In [ ]:
import math

def _nice_round_up(x):
    """Round x up to the nearest 1, 2, or 5 × 10^n."""
    if x <= 0:
        return 1
    mag = 10 ** math.floor(math.log10(x))
    for f in [1, 2, 5, 10]:
        candidate = f * mag
        if candidate >= x:
            return int(candidate)
    return int(x)

def _nice_legend_sizes(max_val):
    """Return [small, mid, top] nice legend sizes where top >= max_val."""
    top   = _nice_round_up(max_val)
    mid   = _nice_round_up(max_val * 0.5)
    small = _nice_round_up(max_val * 0.25)
    return sorted(set([small, mid, top]))

In [ ]:
# Electricity capacity pie charts
# Global max across all periods + existing for consistent circle scaling
sector_max = max(
    all_elec_dfs[case].sum(axis=1).max()
    for case in cases
)
sector_max = max(sector_max,
    all_elec_ex_dfs[cases[0]].sum(axis=1).max(),
    1
)
legend_sizes = _nice_legend_sizes(sector_max)

# Optimized: one map per period
for case in cases:
    plot_province_pies(
        gdf, all_elec_dfs[case],
        colors=custom_color_map,
        pie_scale=7,
        title=f"Electricity Capacity (Optimized) — {scenario_map[case]}",
        legend_sizes=legend_sizes,
        capacity_unit="MW",
        max_total_override=sector_max,
    )

# Existing: one map for the sector
plot_province_pies(
    gdf, all_elec_ex_dfs[cases[0]],
    colors=custom_color_map,
    pie_scale=7,
    title="Electricity Capacity (Existing)",
    legend_sizes=legend_sizes,
    capacity_unit="MW",
    max_total_override=sector_max,
)

In [ ]:
# Steel capacity pie charts
# Global max across all periods + existing for consistent circle scaling
sector_max = max(
    all_steel_dfs[case].sum(axis=1).max()
    for case in cases
)
sector_max = max(sector_max,
    all_steel_ex_dfs[cases[0]].sum(axis=1).max(),
    1
)
legend_sizes = _nice_legend_sizes(sector_max)

# Optimized: one map per period
for case in cases:
    plot_province_pies(
        gdf, all_steel_dfs[case],
        colors=custom_color_map,
        pie_scale=7,
        title=f"Steel Capacity (Optimized) — {scenario_map[case]}",
        legend_sizes=legend_sizes,
        capacity_unit="MT / year",
        max_total_override=sector_max,
    )

# Existing: one map for the sector
ex_data = all_steel_ex_dfs[cases[0]]
if ex_data.values.sum() < 0.001:
    print("No existing steel capacity, skipping")
else:
    plot_province_pies(
        gdf, ex_data,
        colors=custom_color_map,
        pie_scale=7,
        title="Steel Capacity (Existing)",
        legend_sizes=legend_sizes,
        capacity_unit="MT / year",
        max_total_override=sector_max,
    )

In [ ]:
# DRI-making capacity pie charts
# Global max across all periods + existing for consistent circle scaling
_dri_maxes = [
    all_dri_dfs[case].sum(axis=1).max()
    for case in cases
]
_dri_maxes.append(all_dri_ex_dfs[cases[0]].sum(axis=1).max())
sector_max = max(max(_dri_maxes), 1)
legend_sizes = _nice_legend_sizes(sector_max)

# Optimized: one map per period (skip zero-capacity cases)
for case in cases:
    dri_data = all_dri_dfs[case]
    if dri_data.values.sum() < 0.001:
        print(f"No DRI-making capacity: {scenario_map[case]}, skipping")
        continue
    plot_province_pies(
        gdf, dri_data,
        colors=custom_color_map,
        pie_scale=7,
        title=f"DRI-Making Capacity (Optimized) — {scenario_map[case]}",
        legend_sizes=legend_sizes,
        capacity_unit="MT / year",
        max_total_override=sector_max,
    )

# Existing: one map for the sector (skip if no existing DRI)
ex_data = all_dri_ex_dfs[cases[0]]
if ex_data.values.sum() < 0.001:
    print("No existing DRI-making capacity, skipping")
else:
    plot_province_pies(
        gdf, ex_data,
        colors=custom_color_map,
        pie_scale=7,
        title="DRI-Making Capacity (Existing)",
        legend_sizes=legend_sizes,
        capacity_unit="MT / year",
        max_total_override=sector_max,
    )

In [ ]:
# Cement capacity pie charts
# Global max across all periods + existing for consistent circle scaling
sector_max = max(
    all_cement_dfs[case].sum(axis=1).max()
    for case in cases
)
sector_max = max(sector_max,
    all_cement_ex_dfs[cases[0]].sum(axis=1).max(),
    1
)
legend_sizes = _nice_legend_sizes(sector_max)

# Optimized: one map per period
for case in cases:
    plot_province_pies(
        gdf, all_cement_dfs[case],
        colors=custom_color_map,
        pie_scale=4,
        title=f"Cement Capacity (Optimized) — {scenario_map[case]}",
        legend_sizes=[100, 200],
        capacity_unit="MT / year",
        max_total_override=sector_max,
    )

# Existing: one map for the sector
ex_data = all_cement_ex_dfs[cases[0]]
if ex_data.values.sum() < 0.001:
    print("No existing cement capacity, skipping")
else:
    plot_province_pies(
        gdf, ex_data,
        colors=custom_color_map,
        pie_scale=4,
        title="Cement Capacity (Existing)",
        legend_sizes=[100, 200],
        capacity_unit="MT / year",
        max_total_override=sector_max,
    )

In [ ]:
# Aluminum capacity pie charts
# Global max across all periods + existing for consistent circle scaling
sector_max = max(
    all_aluminum_dfs[case].sum(axis=1).max()
    for case in cases
)
sector_max = max(sector_max,
    all_aluminum_ex_dfs[cases[0]].sum(axis=1).max(),
    1
)
legend_sizes = _nice_legend_sizes(sector_max)

# Optimized: one map per period
for case in cases:
    plot_province_pies(
        gdf, all_aluminum_dfs[case],
        colors=custom_color_map,
        pie_scale=5,
        title=f"Aluminum Capacity (Optimized) — {scenario_map[case]}",
        legend_sizes=legend_sizes,
        capacity_unit="MT / year",
        max_total_override=sector_max,
    )

# Existing: one map for the sector
ex_data = all_aluminum_ex_dfs[cases[0]]
if ex_data.values.sum() < 0.001:
    print("No existing aluminum capacity, skipping")
else:
    plot_province_pies(
        gdf, ex_data,
        colors=custom_color_map,
        pie_scale=5,
        title="Aluminum Capacity (Existing)",
        legend_sizes=legend_sizes,
        capacity_unit="MT / year",
        max_total_override=sector_max,
    )

# Material Transport Flows

In [ ]:
def plot_transport_flows(
    flows_df,
    transport_prefix,
    gdf_input,
    province_col="NAME_1",
    title="Transport Flows",
    color="#1f77b4",
    figsize=(12, 10),
    dpi=150,
    flow_threshold=0.005,
    max_shaft_width=150_000,
    min_shaft_width=20_000,
    head_width_ratio=2.5,
    legend_sizes=None,
    unit_label="MT",
    flow_scale=1e6,
    ax=None,
    draw_basemap=True,
    legend_x_anchor="left",
):
    gdf = gdf_input.copy()

    col_pat = re.compile(
        rf"^{re.escape(transport_prefix)}_Region\d+\w+_to_Region\d+\w+_transmission_edge$"
    )
    transport_cols = [c for c in flows_df.columns if col_pat.match(c)]
    if not transport_cols:
        print(f"No columns found for prefix: {transport_prefix}")
        return

    flow_sums = flows_df[transport_cols].sum()

    edge_pat = re.compile(
        rf"^{re.escape(transport_prefix)}_Region\d+(\w+)_to_Region\d+(\w+)_transmission_edge$"
    )
    directed = {}
    for col, val in flow_sums.items():
        m = edge_pat.match(col)
        if m:
            directed[(m.group(1), m.group(2))] = val

    pairs_seen = set()
    net_flows = []
    for (src, dst), fwd in directed.items():
        if (dst, src) in pairs_seen or (src, dst) in pairs_seen:
            continue
        pairs_seen.add((src, dst))
        rev = directed.get((dst, src), 0.0)
        net = (fwd - rev) / flow_scale  # convert to display units (MT)
        if abs(net) > flow_threshold:
            net_flows.append((src, dst, net) if net > 0 else (dst, src, -net))

    if not net_flows:
        print("No net flows above threshold.")
        return

    gdf_proj = gdf.to_crs(epsg=3857)
    gdf_proj["centroid"] = gdf_proj.geometry.centroid
    centroid_map = dict(zip(
        gdf_proj[province_col],
        zip(gdf_proj.centroid.x, gdf_proj.centroid.y)
    ))

    max_flow = max(f for _, _, f in net_flows)

    def shaft_w(flow):
        frac = (flow / max_flow) ** 0.5
        return min_shaft_width + frac * (max_shaft_width - min_shaft_width)

    _own_fig = ax is None
    if _own_fig:
        fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    ax.set_axis_off()

    if draw_basemap:
        gdf_proj.plot(ax=ax, color="#F4F4F4", edgecolor="none", zorder=0)
        gdf_proj.boundary.plot(ax=ax, linewidth=0.3, color="#888888", zorder=1)

    for src, dst, flow in sorted(net_flows, key=lambda x: x[2]):
        if src not in centroid_map or dst not in centroid_map:
            continue
        x0, y0 = centroid_map[src]
        x1, y1 = centroid_map[dst]
        dx, dy = x1 - x0, y1 - y0
        dist = np.hypot(dx, dy)
        sw = shaft_w(flow)
        hw = sw * head_width_ratio
        hl = min(dist * 0.32, hw * 1.4)
        alpha = 0.45 + 0.55 * (flow / max_flow)
        ax.arrow(x0, y0, dx, dy,
                 width=sw, head_width=hw, head_length=hl,
                 length_includes_head=True,
                 fc=color, ec="white", linewidth=0.5,
                 alpha=alpha, zorder=1500)  # above pie wedges (zorder=1000)

    if legend_sizes is None:
        legend_sizes = _nice_legend_sizes(max_flow)

    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    pw = xlim[1] - xlim[0]
    arrow_len = pw * 0.09
    max_legend_sw = shaft_w(max(legend_sizes))
    gap = max_legend_sw * 0.8

    # Expand ylim downward to place legend below the map, never overlapping it
    n_rows = len(legend_sizes) + 1  # arrow rows + title row
    strip_height = gap + n_rows * max_legend_sw * 3.2
    ax.set_ylim(ylim[0] - strip_height, ylim[1])
    ylim = ax.get_ylim()

    lx = (xlim[1] - pw * 0.22) if legend_x_anchor == "right" else (xlim[0] + pw * 0.02)
    ly = ylim[0] + gap

    for size in sorted(legend_sizes):
        sw = shaft_w(size)
        hw = sw * head_width_ratio
        hl = min(arrow_len * 0.32, hw * 1.4)
        ax.arrow(lx, ly, arrow_len, 0,
                 width=sw, head_width=hw, head_length=hl,
                 length_includes_head=True,
                 fc=color, ec="white", linewidth=0.5, alpha=0.85, zorder=2000)
        ax.text(lx + arrow_len + pw * 0.01, ly,
                f"{size:g} {unit_label}",
                va="center", ha="left", fontsize=9, zorder=2001)
        ly += max_legend_sw * 3.2

    ax.text(lx, ly + max_legend_sw * 0.5, "Annual Net Flow",
            fontsize=9, fontweight="bold", zorder=2001)

    if _own_fig:
        plt.title(title, fontsize=16, pad=12)
        plt.tight_layout()
        plt.show()
        plt.close()

    return ax

In [ ]:
transport_prefixes = sorted({
    col.split("_Region", 1)[0]
    for df in all_flow_dfs.values() for col in df.columns
    if "_transport_Region" in col and col.endswith("_transmission_edge")
})
transport_summary_df = pd.DataFrame({
    case: {prefix: df.filter(regex=rf"^{re.escape(prefix)}_.*_transmission_edge$").sum().sum() / 1e6
           for prefix in transport_prefixes}
    for case, df in all_flow_dfs.items()
}).T.rename(index=scenario_map)
transport_summary_df["CO2_Pipeline"] = [
    all_flow_dfs[c].filter(regex=r"(?i)_CO2_Pipeline_transmission_edge$").sum().sum() / 1e6 for c in cases
]
print("Annual gross transport across all links (MT; multi-hop movements count per link):")
display(transport_summary_df)

In [ ]:
# Cement transport — all periods
for case in cases:
    plot_transport_flows(
        flows_df=all_flow_dfs[case],
        transport_prefix="cement_transport",
        gdf_input=gdf,
        title=f"Cement Transport — {scenario_map[case]}",
        color="#e15759",
        figsize=(12, 10),
        legend_sizes=[0.5, 5, 10],
        unit_label="MT",
        flow_scale=1e6,
        flow_threshold=0.001,
    )

In [ ]:
# Scrap steel transport — all periods
for case in cases:
    plot_transport_flows(
        flows_df=all_flow_dfs[case],
        transport_prefix="steelscrap_transport",
        gdf_input=gdf,
        title=f"Scrap Steel Transport — {scenario_map[case]}",
        color="#38a938",
        figsize=(12, 10),
        legend_sizes=[0.5, 2, 5],
        unit_label="MT",
        flow_scale=1e6,
        flow_threshold=0.005,
    )

In [ ]:
# DRI transport — all periods
for case in cases:
    plot_transport_flows(
        flows_df=all_flow_dfs[case],
        transport_prefix="dri_transport",
        gdf_input=gdf,
        title=f"DRI Transport — {scenario_map[case]}",
        color="#9467bd",
        figsize=(12, 10),
        legend_sizes=[0.5, 2, 3],
        unit_label="MT",
        flow_scale=1e6,
        flow_threshold=0.005,
    )

In [ ]:
# Crude steel transport — all periods
for case in cases:
    plot_transport_flows(
        flows_df=all_flow_dfs[case],
        transport_prefix="crudesteel_transport",
        gdf_input=gdf,
        title=f"Crude Steel Transport — {scenario_map[case]}",
        color="#1f77b4",
        figsize=(12, 10),
        legend_sizes=[5, 15, 30],
        unit_label="MT",
        flow_scale=1e6,
        flow_threshold=0.005,
    )

# CO2 Pipeline Transport

In [ ]:
from pyproj import Transformer

BASIN_COORDS = {
    "Songliao":        (124.2489, 45.5424),
    "TurpanHami":      (91.8474,  42.7741),
    "Subei":           (120.2968, 33.2802),
    "BohaiOnshore":    (117.027,  38.1531),
    "Qaidam":          (94.0675,  37.4565),
    "Nanxiang":        (112.3891, 32.6284),
    "Sanjiang":        (132.0,    47.0   ),
    "Hailar":          (118.5403, 48.9011),
    "Jianghan":        (112.9206, 30.4742),
    "Tarim":           (82.2819,  39.3991),
    "Ordos":           (108.7178, 37.5548),
    "YingenEjina":     (101.5,    41.5   ),
    "Hehuai":          (115.0,    34.0   ),
    "Qinshui":         (112.4531, 36.7143),
    "Erlian":          (115.3402, 44.1892),
    "Junggar":         (87.0998,  44.9779),
    "SichuanBasin":    (105.9742, 30.357 ),
    "BohaiOffshore":   (119.7405, 39.0081),
    "NorthYellowSea":  (123.3029, 38.3492),
    "SouthYellowSea":  (123.0614, 35.6424),
    "EastChinaSea":    (124.815,  29.2425),
    "PearlRiverMouth": (114.5827, 20.6148),
    "BeibuGulf":      (109.1529, 20.3578),
    "Qiongdongnan":    (110.6,    17.9   ),
}

def plot_co2_transport_flows(
    flows_df,
    gdf_input,
    province_col="NAME_1",
    title="CO2 Transport Flows",
    color="#4a5568",
    figsize=(13, 11),
    dpi=150,
    flow_threshold=0.1,
    max_shaft_width=80_000,
    min_shaft_width=8_000,
    head_width_ratio=2.5,
    legend_sizes=None,
    unit_label="MT",
    flow_scale=1e6,
):
    transformer = Transformer.from_crs("epsg:4326", "epsg:3857", always_xy=True)
    basin_xy = {name: transformer.transform(lon, lat) for name, (lon, lat) in BASIN_COORDS.items()}

    col_pat = re.compile(r"^(.+?)_to_(.+?)_CO2_Pipeline_transmission_edge$", re.I)
    flows = {}
    for col in flows_df.columns:
        m = col_pat.match(col)
        if m:
            province, basin = (re.sub(r"^R?Region\d+", "", endpoint) for endpoint in m.groups())
            val = flows_df[col].sum() / flow_scale  # convert to display units (MT)
            if val > flow_threshold:
                flows[(province, basin)] = flows.get((province, basin), 0) + val

    if not flows:
        print("No CO2 transport flows found above threshold.")
        return

    gdf = gdf_input.copy().to_crs(epsg=3857)
    gdf["centroid"] = gdf.geometry.centroid
    centroid_map = dict(zip(gdf[province_col], zip(gdf.centroid.x, gdf.centroid.y)))

    endpoint_xy = {**centroid_map, **basin_xy}
    unknown = {x for pair in flows for x in pair if x not in endpoint_xy}
    if unknown:
        warnings.warn(f"Unmapped CO2 pipeline endpoints: {sorted(unknown)}")
    max_flow = max(flows.values())

    def shaft_w(flow):
        frac = (flow / max_flow) ** 0.5
        return min_shaft_width + frac * (max_shaft_width - min_shaft_width)

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    ax.set_axis_off()
    gdf.plot(ax=ax, color="#F4F4F4", edgecolor="none", zorder=0)
    gdf.boundary.plot(ax=ax, linewidth=0.3, color="#888888", zorder=1)

    for (province, basin), flow in sorted(flows.items(), key=lambda x: x[1]):
        if province not in endpoint_xy or basin not in endpoint_xy:
            continue
        x0, y0 = endpoint_xy[province]
        x1, y1 = endpoint_xy[basin]
        dx, dy  = x1 - x0, y1 - y0
        dist    = np.hypot(dx, dy)
        sw = shaft_w(flow)
        hw = sw * head_width_ratio
        hl = min(dist * 0.30, hw * 1.4)
        alpha = 0.30 + 0.70 * (flow / max_flow)
        ax.arrow(x0, y0, dx, dy,
                 width=sw, head_width=hw, head_length=hl,
                 length_includes_head=True,
                 fc=color, ec="white", linewidth=0.4,
                 alpha=alpha, zorder=5)

    for name, (bx, by) in basin_xy.items():
        ax.plot(bx, by, "v", color="#e15759", markersize=6, zorder=10, clip_on=False)
        ax.text(bx, by + 50_000, name, fontsize=6, ha="center", va="bottom", zorder=11, clip_on=False)

    plt.title(title, fontsize=16, pad=12)

    if legend_sizes is None:
        legend_sizes = _nice_legend_sizes(max_flow)

    xlim = ax.get_xlim(); ylim = ax.get_ylim()
    pw = xlim[1] - xlim[0]
    arrow_len = pw * 0.09
    lx = xlim[0] + pw * 0.02
    max_legend_sw = shaft_w(max(legend_sizes))
    gap = max_legend_sw * 0.8

    # Expand ylim downward to place legend below the map, never overlapping it
    n_rows = len(legend_sizes) + 1
    strip_height = gap + n_rows * max_legend_sw * 3.2
    ax.set_ylim(ylim[0] - strip_height, ylim[1])
    ylim = ax.get_ylim()
    ly = ylim[0] + gap

    for size in sorted(legend_sizes):
        sw = shaft_w(size)
        hw = sw * head_width_ratio
        hl = min(arrow_len * 0.32, hw * 1.4)
        ax.arrow(lx, ly, arrow_len, 0,
                 width=sw, head_width=hw, head_length=hl,
                 length_includes_head=True,
                 fc=color, ec="white", linewidth=0.5, alpha=0.85, zorder=10)
        ax.text(lx + arrow_len + pw * 0.01, ly,
                f"{size:g} {unit_label}", va="center", ha="left", fontsize=9, zorder=11)
        ly += max_legend_sw * 3.2

    ax.text(lx, ly + max_legend_sw * 0.5, "Annual Flow",
            fontsize=9, fontweight="bold", zorder=11)

    plt.tight_layout()
    plt.show()
    plt.close()

In [ ]:
# CO2 pipeline transport — all periods
for case in cases:
    plot_co2_transport_flows(
        flows_df=all_flow_dfs[case],
        gdf_input=gdf,
        title=f"CO2 Pipeline Transport — {scenario_map[case]}",
        color="#4a5568",
        figsize=(13, 11),
        legend_sizes=[0.5, 2, 5],
        unit_label="MT",
        flow_scale=1e6,
        flow_threshold=0.1,
    )

In [ ]:
# ── Sector lookup tables ────────────────────────────────────────────────────
_sector_capacity_dfs = {
    "electricity": all_elec_dfs,
    "steel":       all_steel_dfs,
    "cement":      all_cement_dfs,
    "aluminum":    all_aluminum_dfs,
    "dri":         all_dri_dfs,
}
_sector_units = {
    "electricity": "MW",
    "steel":       "MT / year",
    "cement":      "MT / year",
    "aluminum":    "MT / year",
    "dri":         "MT / year",
}

def plot_capacity_with_flows(
    gdf,
    capacity_df,
    flows_df,
    transport_prefix,
    capacity_colors=None,
    pie_scale=1.0,
    figsize=(14, 10),
    dpi=150,
    title="",
    capacity_legend_sizes=None,
    capacity_unit="MW",
    max_total_override=None,
    transport_color="#1f77b4",
    flow_threshold=0.005,
    max_shaft_width=150_000,
    min_shaft_width=20_000,
    head_width_ratio=2.5,
    transport_legend_sizes=None,
    transport_unit_label="MT",
    transport_flow_scale=1e6,
):
    """Plot sector capacity pies and transport flows on the same map."""
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

    plot_province_pies(
        gdf, capacity_df,
        colors=capacity_colors,
        pie_scale=pie_scale,
        legend_sizes=capacity_legend_sizes,
        capacity_unit=capacity_unit,
        max_total_override=max_total_override,
        ax=ax,
    )

    plot_transport_flows(
        flows_df=flows_df,
        transport_prefix=transport_prefix,
        gdf_input=gdf,
        color=transport_color,
        flow_threshold=flow_threshold,
        max_shaft_width=max_shaft_width,
        min_shaft_width=min_shaft_width,
        head_width_ratio=head_width_ratio,
        legend_sizes=transport_legend_sizes,
        unit_label=transport_unit_label,
        flow_scale=transport_flow_scale,
        ax=ax,
        draw_basemap=False,
        legend_x_anchor="right",
    )

    plt.title(title, fontsize=18)
    plt.tight_layout()
    plt.show()
    plt.close()

# Capacity + Transport Flow Overlay

In [ ]:
# Select period_1, period_2, period_3, or period_4 for the overlays below.
plot_case = "period_4"
plot_sector = "steel"
plot_transport_prefix = "crudesteel_transport"
plot_transport_color = "#1f77b4"
plot_transport_unit = "MT"
plot_flow_threshold = 0.005

In [ ]:
plot_sector = "steel"
plot_transport_prefix = "crudesteel_transport"

capacity_df = _sector_capacity_dfs[plot_sector][plot_case]

# Sector-wide max across all periods for consistent circle scaling
sector_max = max(
    _sector_capacity_dfs[plot_sector][c].sum(axis=1).max()
    for c in cases
)
sector_max = max(sector_max, 1)

plot_capacity_with_flows(
    gdf=gdf,
    capacity_df=capacity_df,
    flows_df=all_flow_dfs[plot_case],
    transport_prefix=plot_transport_prefix,
    capacity_colors=custom_color_map,
    pie_scale=7,
    figsize=(14, 10),
    title=(
        f"{plot_sector.title()} Capacity + "
        f"{plot_transport_prefix.replace('_', ' ').title()} — "
        f"{scenario_map[plot_case]}"
    ),
    capacity_legend_sizes=_nice_legend_sizes(sector_max),
    capacity_unit=_sector_units[plot_sector],
    max_total_override=sector_max,
    transport_color=plot_transport_color,
    transport_unit_label=plot_transport_unit,
    flow_threshold=plot_flow_threshold,
)

In [ ]:
plot_sector = "steel"
plot_transport_prefix = "steelscrap_transport"

capacity_df = _sector_capacity_dfs[plot_sector][plot_case]

# Sector-wide max across all periods for consistent circle scaling
sector_max = max(
    _sector_capacity_dfs[plot_sector][c].sum(axis=1).max()
    for c in cases
)
sector_max = max(sector_max, 1)

plot_capacity_with_flows(
    gdf=gdf,
    capacity_df=capacity_df,
    flows_df=all_flow_dfs[plot_case],
    transport_prefix=plot_transport_prefix,
    capacity_colors=custom_color_map,
    pie_scale=7,
    figsize=(14, 10),
    title=(
        f"{plot_sector.title()} Capacity + "
        f"{plot_transport_prefix.replace('_', ' ').title()} — "
        f"{scenario_map[plot_case]}"
    ),
    capacity_legend_sizes=_nice_legend_sizes(sector_max),
    capacity_unit=_sector_units[plot_sector],
    max_total_override=sector_max,
    transport_color="forestgreen",
    transport_unit_label=plot_transport_unit,
    flow_threshold=plot_flow_threshold,
)

In [ ]:
plot_sector = "dri"
plot_transport_prefix = "dri_transport"

capacity_df = _sector_capacity_dfs[plot_sector][plot_case]

# Sector-wide max across all periods for consistent circle scaling
sector_max = max(
    _sector_capacity_dfs[plot_sector][c].sum(axis=1).max()
    for c in cases
)
sector_max = max(sector_max, 1)

plot_capacity_with_flows(
    gdf=gdf,
    capacity_df=capacity_df,
    flows_df=all_flow_dfs[plot_case],
    transport_prefix=plot_transport_prefix,
    capacity_colors=custom_color_map,
    pie_scale=2.5,
    figsize=(14, 10),
    title=(
        f"{plot_sector.title()} Capacity + "
        f"{plot_transport_prefix.replace('_', ' ').title()} — "
        f"{scenario_map[plot_case]}"
    ),
    capacity_legend_sizes=_nice_legend_sizes(sector_max),
    capacity_unit=_sector_units[plot_sector],
    max_total_override=sector_max,
    transport_color=plot_transport_color,
    transport_unit_label=plot_transport_unit,
    flow_threshold=plot_flow_threshold,
)

In [ ]:
plot_sector = "cement"
plot_transport_prefix = "cement_transport"
plot_case = "period_1"

capacity_df = _sector_capacity_dfs[plot_sector][plot_case]

# Sector-wide max across all periods for consistent circle scaling
sector_max = max(
    _sector_capacity_dfs[plot_sector][c].sum(axis=1).max()
    for c in cases
)
sector_max = max(sector_max, 1)

plot_capacity_with_flows(
    gdf=gdf,
    capacity_df=capacity_df,
    flows_df=all_flow_dfs[plot_case],
    transport_prefix=plot_transport_prefix,
    capacity_colors=custom_color_map,
    pie_scale=2.5,
    figsize=(14, 10),
    title=(
        f"{plot_sector.title()} Capacity + "
        f"{plot_transport_prefix.replace('_', ' ').title()} — "
        f"{scenario_map[plot_case]}"
    ),
    capacity_legend_sizes=_nice_legend_sizes(sector_max),
    capacity_unit=_sector_units[plot_sector],
    max_total_override=sector_max,
    transport_color="gray",
    transport_unit_label=plot_transport_unit,
    flow_threshold=plot_flow_threshold,
)

In [ ]:
plot_sector = "cement"
plot_transport_prefix = "cement_transport"
plot_case = "period_4"

capacity_df = _sector_capacity_dfs[plot_sector][plot_case]

# Sector-wide max across all periods for consistent circle scaling
sector_max = max(
    _sector_capacity_dfs[plot_sector][c].sum(axis=1).max()
    for c in cases
)
sector_max = max(sector_max, 1)

plot_capacity_with_flows(
    gdf=gdf,
    capacity_df=capacity_df,
    flows_df=all_flow_dfs[plot_case],
    transport_prefix=plot_transport_prefix,
    capacity_colors=custom_color_map,
    pie_scale=2.5,
    figsize=(14, 10),
    title=(
        f"{plot_sector.title()} Capacity + "
        f"{plot_transport_prefix.replace('_', ' ').title()} — "
        f"{scenario_map[plot_case]}"
    ),
    capacity_legend_sizes=_nice_legend_sizes(sector_max),
    capacity_unit=_sector_units[plot_sector],
    max_total_override=sector_max,
    transport_color="gray",
    transport_unit_label=plot_transport_unit,
    flow_threshold=plot_flow_threshold,
)

In [ ]:
for sector in ("alumina", "ammonia", "methanol", "hydrogen"):
    unit = "MW" if sector == "hydrogen" else "MT / year"
    plot_stacked_bar(consolidate_capacity(sector_capacity[sector]).T,
                     scenario_map=scenario_map, color_map=custom_color_map,
                     title_name=f"{sector.title()} Capacity by Period",
                     xaxis_name="Period", yaxis_name=f"Capacity ({unit})").show()
    maximum = max(1, *(df.sum(axis=1).max() for df in sector_capacity[sector].values()))
    for case in cases:
        plot_province_pies(gdf, sector_capacity[sector][case], colors=custom_color_map,
                          pie_scale=7, title=f"{sector.title()} Capacity — {scenario_map[case]}",
                          capacity_unit=unit, max_total_override=maximum,
                          legend_sizes=_nice_legend_sizes(maximum))

for prefix in transport_prefixes:
    if prefix in {"cement_transport", "steelscrap_transport", "dri_transport", "crudesteel_transport"}:
        continue  # Already plotted above.
    for case in cases:
        plot_transport_flows(all_flow_dfs[case], prefix, gdf,
                             title=f"{prefix.replace('_', ' ').title()} — {scenario_map[case]}")